# Day 1 Lab - Windows-Style Log Generator & Elastic Cloud Uploader
### AIOps Fundamentals Training | Microland

---

## What This Notebook Does

This notebook simulates the operational log data an IT team collects in a real environment.  
It generates **1,000 Windows-style log entries** spanning the last 4 hours, with **20 injected anomalies**
hidden inside the normal traffic - exactly like a brute-force attempt buried in a busy Windows event log.

Once generated, all data is uploaded directly to your **Elastic Cloud** deployment.

---

## Learning Objectives
- Understand what normalised ECS-aligned operational log data looks like
- See how 'normal' log patterns differ from anomalous patterns
- Practice navigating event data in Kibana Discover before ML is applied

---

## Before You Start
1. You have a running **Elastic Cloud** deployment (created in the Day 1 lab)
2. You have your **Cloud ID** and **API Key** ready (from the Elastic Cloud console)
3. You are running this in **Google Colab** (no local install needed)

> **No Python knowledge required.** Run each cell in order using the Play button or Shift+Enter.


## Step 1 - Install Required Libraries
This installs the Elasticsearch Python client. Takes about 15 seconds.


In [ ]:
!pip install elasticsearch --quiet
print('Libraries installed successfully.')


## Step 2 - Enter Your Elastic Cloud Connection Details

**Where to find these values:**
1. Go to cloud.elastic.co and open your deployment
2. **Cloud ID**: shown on the deployment overview page - click Copy
3. **API Key**: go to Stack Management > API Keys > Create API Key - copy the encoded key

> **Important:** Never share your API key. Treat it like a password.


In [ ]:
# -----------------------------------------------------------
# CONFIGURATION - Fill in your own values here
# -----------------------------------------------------------

CLOUD_ID   = 'YOUR ELASTIC CLOUD ID'   # Paste your Elastic Cloud ID
API_KEY    = 'YOUR ELASTIC CLOUD API' # Paste your Elastic API Key
INDEX_NAME = 'aiops-lab-log-generator-day1'       # Index to create in Elasticsearch

print(f'Configuration set.')
print(f'   Target index : {INDEX_NAME}')


## Step 3 - Connect to Elastic Cloud
This cell tests that your credentials are correct before uploading any data.


In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    cloud_id=CLOUD_ID,
    api_key=API_KEY
)

info = es.info()
print(f'Connected to Elasticsearch cluster: {info["cluster_name"]}')
print(f'   Elasticsearch version : {info["version"]["number"]}')


## Step 4 - Create the Index with ECS-Aligned Mapping

We define the field structure (called a **mapping**) before uploading data.  
The field names below mirror the **Elastic Common Schema (ECS)** - the same schema
that Winlogbeat and Filebeat use when shipping real Windows events.

> **AIOps Concept:** Without a consistent schema, ML cannot compare events from different sources.
> The mapping below enforces consistency so our simulated data behaves like real production data.


In [ ]:
# Delete index if it already exists (safe for lab re-runs)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f'Deleted existing index: {INDEX_NAME}')

mapping = {
    'mappings': {
        'properties': {
            '@timestamp':          {'type': 'date'},
            'event.action':        {'type': 'keyword'},
            'event.category':      {'type': 'keyword'},
            'event.outcome':       {'type': 'keyword'},
            'event.code':          {'type': 'keyword'},
            'event.dataset':       {'type': 'keyword'},
            'host.name':           {'type': 'keyword'},
            'host.os.name':        {'type': 'keyword'},
            'user.name':           {'type': 'keyword'},
            'source.ip':           {'type': 'ip'},
            'source.port':         {'type': 'integer'},
            'message':             {'type': 'text'},
            'log.level':           {'type': 'keyword'},
            'is_anomaly':          {'type': 'boolean'},
            'anomaly_description': {'type': 'text'}
        }
    }
}

es.indices.create(index=INDEX_NAME, body=mapping)
print(f'Index "{INDEX_NAME}" created with ECS-aligned mapping.')


## Step 5 - Generate 1,000 Log Events

This cell builds the dataset:

| Event Type | Count | Description |
|------------|-------|-------------|
| Normal logon events (Event ID 4624) | ~500 | Successful logins from known internal IPs |
| Normal failed logons (Event ID 4625) | ~200 | Occasional failed logins - normal background noise |
| Service events (Event ID 7036) | ~200 | Services starting and stopping - routine activity |
| Process creation (Event ID 4688) | ~80 | New processes created - normal admin activity |
| **Injected anomalies** (Event ID 4625) | **20** | Rapid failed logins from one suspicious external IP |

> **Your challenge in Kibana:** Can you find the 20 anomalous events hidden in the 980 normal ones?


In [ ]:
import random
import json
from datetime import datetime, timedelta, timezone

random.seed(42)  # Fixed seed for reproducibility

# Time range: last 4 hours
now        = datetime.now(timezone.utc)
start_time = now - timedelta(hours=4)

# Reference data
SERVERS      = ['WINTEL-SRV-01', 'WINTEL-SRV-02', 'WINTEL-SRV-03',
                 'DC-PROD-01', 'DC-PROD-02', 'APP-SRV-07', 'FILE-SRV-03']
INTERNAL_IPS = ['10.10.1.10', '10.10.1.11', '10.10.1.25',
                 '192.168.1.50', '192.168.1.51', '192.168.10.20']
DOMAIN_USERS = ['jsmith', 'aparikh', 'nkumar', 'rdesai', 'srao',
                 'admin_it', 'svc_backup', 'svc_monitor']
PROCESSES    = ['svchost.exe', 'lsass.exe', 'explorer.exe', 'powershell.exe',
                 'cmd.exe', 'mmc.exe', 'taskmgr.exe']
SERVICES     = ['Windows Update', 'Print Spooler', 'DHCP Client',
                 'DNS Client', 'Windows Defender', 'Remote Desktop Services']

# Anomaly: 20 rapid failed logins from one suspicious external IP in a 3-minute window
ANOMALY_IP   = '185.220.101.47'   # Known Tor exit-node range
ANOMALY_USER = 'administrator'
anomaly_start = start_time + timedelta(hours=2, minutes=15)
anomaly_end   = anomaly_start + timedelta(minutes=3)


def rand_ts(t0, t1):
    return t0 + timedelta(seconds=random.uniform(0, (t1 - t0).total_seconds()))


def ev_logon_ok(ts):
    u, ip, h = random.choice(DOMAIN_USERS), random.choice(INTERNAL_IPS), random.choice(SERVERS)
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'logged-in',
        'event.category': 'authentication', 'event.outcome': 'success',
        'event.code': '4624', 'event.dataset': 'system.security',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': u, 'source.ip': ip, 'source.port': random.randint(49152, 65535),
        'log.level': 'information',
        'message': f'An account was successfully logged on. Account: {u}, IP: {ip}',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_logon_fail(ts, ip=None, user=None):
    u, src, h = (user or random.choice(DOMAIN_USERS)), (ip or random.choice(INTERNAL_IPS)), random.choice(SERVERS)
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'authentication_failure',
        'event.category': 'authentication', 'event.outcome': 'failure',
        'event.code': '4625', 'event.dataset': 'system.security',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': u, 'source.ip': src, 'source.port': random.randint(49152, 65535),
        'log.level': 'warning',
        'message': f'An account failed to log on. Account: {u}, IP: {src}, Reason: Unknown user name or bad password',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_service(ts):
    svc, h, state = random.choice(SERVICES), random.choice(SERVERS), random.choice(['started', 'stopped'])
    return {
        '@timestamp': ts.isoformat(), 'event.action': f'service-{state}',
        'event.category': 'process', 'event.outcome': 'success',
        'event.code': '7036', 'event.dataset': 'system.system',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': 'SYSTEM', 'source.ip': '127.0.0.1', 'source.port': 0,
        'log.level': 'information',
        'message': f'The {svc} service entered the {state} state.',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_process(ts):
    proc, user, h = random.choice(PROCESSES), random.choice(DOMAIN_USERS), random.choice(SERVERS)
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'process_started',
        'event.category': 'process', 'event.outcome': 'success',
        'event.code': '4688', 'event.dataset': 'system.security',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': user, 'source.ip': random.choice(INTERNAL_IPS), 'source.port': 0,
        'log.level': 'information',
        'message': f'A new process was created. Name: {proc}, Creator: {user}',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_anomaly(ts):
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'authentication_failure',
        'event.category': 'authentication', 'event.outcome': 'failure',
        'event.code': '4625', 'event.dataset': 'system.security',
        'host.name': 'DC-PROD-01', 'host.os.name': 'Windows Server 2019',
        'user.name': ANOMALY_USER, 'source.ip': ANOMALY_IP,
        'source.port': random.randint(49152, 65535),
        'log.level': 'warning',
        'message': f'An account failed to log on. Account: {ANOMALY_USER}, IP: {ANOMALY_IP}, Reason: Unknown user name or bad password',
        'is_anomaly': True,
        'anomaly_description': 'Brute-force: 20 rapid failed logins from external IP 185.220.101.47 against administrator in a 3-minute window'
    }


# Build dataset
events = []
for _ in range(500): events.append(ev_logon_ok(rand_ts(start_time, now)))
for _ in range(200): events.append(ev_logon_fail(rand_ts(start_time, now)))
for _ in range(200): events.append(ev_service(rand_ts(start_time, now)))
for _ in range(80):  events.append(ev_process(rand_ts(start_time, now)))
for _ in range(20):  events.append(ev_anomaly(rand_ts(anomaly_start, anomaly_end)))

events.sort(key=lambda x: x['@timestamp'])

normal_count  = sum(1 for e in events if not e['is_anomaly'])
anomaly_count = sum(1 for e in events if     e['is_anomaly'])

print(f'Generated {len(events):,} log events')
print(f'   Normal events  : {normal_count:,}')
print(f'   Anomaly events : {anomaly_count:,}  <- hidden in the stream')
print(f'   Time range     : {events[0]["@timestamp"][:19]}  to  {events[-1]["@timestamp"][:19]}')
print(f'   Anomaly window : {anomaly_start.strftime("%H:%M:%S")}  to  {anomaly_end.strftime("%H:%M:%S")} UTC')
print()
print('Sample - most recent event:')
print(json.dumps(events[-1], indent=2))


## Step 6 - Upload to Elastic Cloud

All 1,000 events are uploaded in one efficient batch using the Elasticsearch bulk API.


In [ ]:
from elasticsearch.helpers import bulk

def generate_actions(docs, index):
    for doc in docs:
        yield {'_index': index, '_source': doc}

print(f'Uploading {len(events):,} events to index "{INDEX_NAME}" ...')

success, errors = bulk(
    es,
    generate_actions(events, INDEX_NAME),
    chunk_size=200,
    raise_on_error=False
)

print(f'Upload complete!')
print(f'   Documents indexed : {success:,}')
if errors:
    print(f'   Errors            : {len(errors)}')
    for e in errors[:3]: print(f'      {e}')
else:
    print(f'   Errors            : 0')

es.indices.refresh(index=INDEX_NAME)
print('Index refreshed - data is now searchable in Kibana.')


## Step 7 - Verify the Upload


In [ ]:
total   = es.count(index=INDEX_NAME)['count']
anomaly = es.count(index=INDEX_NAME, body={'query': {'term': {'is_anomaly': True}}})['count']

agg = es.search(index=INDEX_NAME, body={
    'size': 0,
    'aggs': {'by_action': {'terms': {'field': 'event.action', 'size': 10}}}
})

print(f'Verification summary - index "{INDEX_NAME}":')
print(f'   Total documents : {total:,}')
print(f'   Anomaly events  : {anomaly:,}  <- try to find these in Kibana!')
print()
print('Event breakdown by action:')
for b in agg['aggregations']['by_action']['buckets']:
    print(f'   {b["key"]:35s}  {b["doc_count"]:>5,} events')


## Step 8 - Explore Your Data in Kibana

### 8a - Open Kibana Discover
1. Go to your Elastic Cloud deployment and click **Launch Kibana**
2. In the left navigation menu, click **Discover**
3. If prompted to create a data view:
   - Index pattern: `aiops-lab-day2`
   - Timestamp field: `@timestamp`
   - Click **Save data view to Kibana**

---

### 8b - KQL Queries to Try in Kibana

Copy and paste these into the Kibana search bar:

```
# Show only failed logons
event.action : "authentication_failure"

# Show only the injected anomalies
is_anomaly : true

# Show events from the suspicious IP
source.ip : "185.220.101.47"

# Combine filters
event.action : "authentication_failure" AND source.ip : "185.220.101.47"
```

---



---

### Challenge Question

> The 20 anomalous events from IP `185.220.101.47` are hidden among 200 normal failed logons.
> Can you spot them visually in your bar chart or timeline?
> **This is exactly why we need ML** - human eyes cannot reliably detect low-volume,
> time-clustered anomalies in dashboards full of noise.

---

### Lab Complete
In Day 2 you will build an ML job to detect anomalies like this automatically.


## Step 8 - Build Dashboard
### 8c - Build the Dashboard (Lab Task)

Go to **Dashboards > Create dashboard > Add visualisation** and create:

| Panel | Type | Field |
|-------|------|-------|
| CPU utilization | Line chart | `system.cpu.pct` |
| Memory Utilization | Line Chart | 'system.memory.used_gb' |
| System Status | Bar chart | `system.status` |
| Host Region | Pie Chart | `host.region` |

---




# Before creating Dashboard Run the following code block

In [ ]:
import datetime
import random
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

INDEX_NAME = "example-telemetry-for-dashboard"

print("Connecting to Elastic Cloud...")
es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)

# Clear old data if it exists so students start fresh
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"Cleared existing index: {INDEX_NAME}")

# Define servers and application environments
servers = [
    {"name": "ml-prod-web-01", "env": "Production", "role": "Web Server"},
    {"name": "ml-prod-db-01", "env": "Production", "role": "Database"},
    {"name": "ml-stage-app-01", "env": "Staging", "role": "Application"}
]

services = ["IIS W3SVC", "PostgreSQL", "Active Directory", "Docker Engine"]
regions = ["APAC-Bangalore", "AMER-Texas", "EMEA-London"]

actions = []
now = datetime.datetime.utcnow()

print("Generating 24 hours of infrastructure metrics...")
# Loop backwards through 24 hours in 5-minute intervals (288 data points per server)
for minutes_ago in range(0, 24 * 60, 5):
    record_time = now - datetime.timedelta(minutes=minutes_ago)
    timestamp_str = record_time.isoformat() + "Z"

    for s in servers:
        # 1. Base Metrics
        cpu = random.uniform(15.0, 45.0)
        # Allocate max memory capacity based on server role
        mem_capacity = 64.0 if s["role"] == "Database" else 32.0
        mem_used = random.uniform(mem_capacity * 0.3, mem_capacity * 0.5)
        net_in = random.uniform(50.0, 250.0)
        net_out = random.uniform(80.0, 400.0)
        status = "Healthy"
        response_time = random.randint(10, 85)

        # 2. Inject an Incident/Spike Simulation (Happened between 2 and 4 hours ago)
        # Specifically targeting the production database node
        if 120 <= minutes_ago <= 240 and s["name"] == "ml-prod-db-01":
            cpu = random.uniform(88.0, 99.5)                  # Severe CPU Spike
            mem_used = random.uniform(mem_capacity * 0.88, mem_capacity * 0.96) # Out of memory territory
            net_in = random.uniform(800.0, 1200.0)            # High incoming backup data
            status = "Warning" if minutes_ago > 180 else "Critical"
            response_time = random.randint(450, 1200)         # Degraded performance

        # 3. Add random service errors to look good on Pie charts
        log_level = "INFO"
        message = "System health check optimal."
        if status == "Critical":
            log_level = "CRITICAL"
            message = "Resource exhaustion detected: Thread pool deadlocked under heavy memory pressure."
        elif random.random() < 0.03: # 3% chance of random warnings on healthy nodes
            log_level = "WARNING"
            message = f"Service [{random.choice(services)}] experienced a brief query delay reset."

        # Document Payload Structure
        doc = {
            "_index": INDEX_NAME,
            "_source": {
                "@timestamp": timestamp_str,
                "host.name": s["name"],
                "host.environment": s["env"],
                "host.role": s["role"],
                "host.region": random.choice(regions),
                "system.cpu.pct": round(cpu, 2),
                "system.memory.capacity_gb": mem_capacity,
                "system.memory.used_gb": round(mem_used, 2),
                "system.network.in_mbps": round(net_in, 2),
                "system.network.out_mbps": round(net_out, 2),
                "service.response_time_ms": response_time,
                "system.status": status,
                "log.level": log_level,
                "message": message
            }
        }
        actions.append(doc)

# Bulk ingest into Elasticsearch
print(f"Uploading {len(actions)} metric telemetry documents...")
bulk(es, actions)
es.indices.refresh(index=INDEX_NAME)
print("Telemetry upload complete! Instruct your students to create their Data View now.")

# Simulation of Realtime Telemitry

In [ ]:
import datetime
import random
import time
from elasticsearch import Elasticsearch

# --- CONFIGURATION ---

INDEX_NAME = "example-telemetry-for-dashboard"
INTERVAL_SECONDS = 30  # Stream a new data point every 30 seconds

print("Connecting to Elastic Cloud...")
es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)

# Define infrastructure nodes
servers = [
    {"name": "ml-prod-web-01", "env": "Production", "role": "Web Server"},
    {"name": "ml-prod-db-01", "env": "Production", "role": "Database"},
    {"name": "ml-stage-app-01", "env": "Staging", "role": "Application"}
]

services = ["IIS W3SVC", "PostgreSQL", "Active Directory", "Docker Engine"]
regions = ["APAC-Bangalore", "AMER-Texas", "EMEA-London"]

print(f"Starting real-time log streaming into index: [{INDEX_NAME}]")
print(f"Data will be appended every {INTERVAL_SECONDS} seconds. Press STOP in Colab to terminate.")
print("-" * 60)

iteration_count = 0

try:
    while True:
        # Use the exact current moment in UTC for true live streaming
        now = datetime.datetime.utcnow()
        timestamp_str = now.isoformat() + "Z"
        iteration_count += 1

        print(f"[{now.strftime('%H:%M:%S')}] Generating metric batch #{iteration_count}...")

        for s in servers:
            # 1. Standard Baseline Performance Metrics
            cpu = random.uniform(12.0, 38.0)
            mem_capacity = 64.0 if s["role"] == "Database" else 32.0
            mem_used = random.uniform(mem_capacity * 0.25, mem_capacity * 0.40)
            net_in = random.uniform(45.0, 180.0)
            net_out = random.uniform(70.0, 290.0)
            status = "Healthy"
            response_time = random.randint(12, 60)
            log_level = "INFO"
            message = "System health check optimal."

            # 2. Inject a Live Incident Event Lifecycle (Simulated after a few iterations)
            # This causes the database node's metrics to spike live on the dashboard!
            if 10 <= iteration_count <= 25 and s["name"] == "ml-prod-db-01":
                cpu = random.uniform(85.0, 98.9)                                    # Severe CPU Spike
                mem_used = random.uniform(mem_capacity * 0.85, mem_capacity * 0.94)   # High Memory pressure
                net_in = random.uniform(750.0, 1100.0)                                # Mass data ingress spike
                status = "Critical" if iteration_count >= 15 else "Warning"
                response_time = random.randint(350, 900)
                log_level = "CRITICAL" if status == "Critical" else "WARNING"
                message = "Resource crunch: Thread synchronization bottleneck detected in active connections pool."

            # Build the document payload
            doc = {
                "@timestamp": timestamp_str,
                "host.name": s["name"],
                "host.environment": s["env"],
                "host.role": s["role"],
                "host.region": random.choice(regions),
                "system.cpu.pct": round(cpu, 2),
                "system.memory.capacity_gb": mem_capacity,
                "system.memory.used_gb": round(mem_used, 2),
                "system.network.in_mbps": round(net_in, 2),
                "system.network.out_mbps": round(net_out, 2),
                "service.response_time_ms": response_time,
                "system.status": status,
                "log.level": log_level,
                "message": message
            }

            # Index the document live
            es.index(index=INDEX_NAME, document=doc)

        # Refresh the index instantly so dashboards reflect the data points within the same second
        es.indices.refresh(index=INDEX_NAME)
        print(f"      Successfully sent data batch #{iteration_count} to Elastic Cloud.")

        # Pause execution for the specified interval window
        time.sleep(INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\nReal-time telemetry stream stopped cleanly by user.")